# Join Price/MSP Data with Mandi Master

This notebook merges the cleaned price/MSP dataset with the mandi master using `mandi_id` first and `district` as a fallback. It also validates the match quality before exporting the joined file.

In [ ]:
import pandas as pd
from pathlib import Path

root = Path(r"c:\Users\Manya\Downloads\track3_agritech_dataset_files")
price_path = root / "cleaned_price_and_msp.csv"
master_path = root / "cleaned_mandi_master.csv"
output_path = root / "price_and_msp_joined_mandi_master.csv"

price_df = pd.read_csv(price_path)
master_df = pd.read_csv(master_path)

print("price rows:", len(price_df))
print("master rows:", len(master_df))
print(price_df.head())
print(master_df.head())

In [ ]:
# Standardize key columns for matching
price_df["mandi_id"] = price_df["mandi_id"].astype("string").str.strip().str.upper()
price_df["district"] = price_df["district"].astype("string").str.strip().str.title()

master_df["Mandi_ID"] = master_df["Mandi_ID"].astype("string").str.strip().str.upper()
master_df["District"] = master_df["District"].astype("string").str.strip().str.title()

# Build lookup tables
master_by_id = master_df[["Mandi_ID", "Mandi_Name", "District", "State", "Mandi_Type", "Total_Area_Acres"]].drop_duplicates()
master_by_id = master_by_id.rename(columns={"Mandi_ID": "mandi_id"})

master_by_district = master_df[["District", "Mandi_Name", "State", "Mandi_Type", "Total_Area_Acres"]].drop_duplicates()
master_by_district = master_by_district.rename(columns={
    "District": "district",
    "Mandi_Name": "Mandi_Name_district",
    "State": "State_district",
    "Mandi_Type": "Mandi_Type_district",
    "Total_Area_Acres": "Total_Area_Acres_district",
})

# Join in two stages: exact mandi_id match first, district fallback second
joined_df = price_df.merge(master_by_id, on="mandi_id", how="left")
joined_df = joined_df.merge(master_by_district, on="district", how="left")

# Fill missing master fields from district match only where mandi_id match did not exist
for col, fallback in [
    ("Mandi_Name", "Mandi_Name_district"),
    ("State", "State_district"),
    ("Mandi_Type", "Mandi_Type_district"),
    ("Total_Area_Acres", "Total_Area_Acres_district"),
]:
    joined_df[col] = joined_df[col].combine_first(joined_df[fallback])

# Add explicit master columns for readability
joined_df["Mandi_ID"] = joined_df["mandi_id"]
joined_df["District"] = joined_df["district"]

print("joined sample:")
print(joined_df.head())

In [ ]:
# Validation summary before export
matched_by_mandi_id = joined_df["Mandi_Name"].notna().sum()
matched_by_district = joined_df["Mandi_Name_district"].notna().sum()
total_rows = len(joined_df)

print("Total rows:", total_rows)
print("Matched by mandi_id:", matched_by_mandi_id)
print("Matched by district fallback:", matched_by_district)
print("Mandi_id match %:", round((matched_by_mandi_id / total_rows) * 100, 2), "%")
print("District fallback match %:", round((matched_by_district / total_rows) * 100, 2), "%")

# Quick validation checks
print("Missing master metadata rows:", joined_df["Mandi_Name"].isna().sum())
print("Duplicate rows in joined dataset:", joined_df.duplicated().sum())

# Export final joined dataset
final_columns = [
    "record_id", "date", "mandi_id", "district", "crop_name",
    "min_price", "max_price", "modal_price", "msp",
    "Mandi_ID", "Mandi_Name", "District", "State", "Mandi_Type", "Total_Area_Acres"
]

output_df = joined_df[final_columns].copy()
output_df.to_csv(output_path, index=False)
print("Joined dataset exported to:", output_path)